# 05 — Field Extraction

## Purpose
Extracts structured field values from each classified document segment using
`AI_COMPLETE`. Each segment (`CHILD_DOC_ID`) is processed independently
using the field schema defined in `DOC_TYPE_CONFIG` for its document type,
producing one extracted value per field with a self-reported confidence score.
Results are written to `DOCUMENTS_EXTRACTED` (one row per segment) and
`DOCUMENTS_EXTRACTED_FLAT` (one row per field per segment).

## What this notebook does
Loads active field schemas from `DOC_TYPE_CONFIG` for all doc types pending
extraction. For each doc type a single SQL call runs `AI_COMPLETE` across
all unextracted segments in parallel — Snowflake handles row-level
parallelism natively. The extraction prompt is built
from the field questions in `DOC_TYPE_CONFIG` and injected with each
segment's concatenated page text via `REPLACE()` inside the SQL statement.

Results are parsed in Python and written to both output tables. Token
consumption (estimated from character count) is logged to `LLM_USAGE` for
PoC cost reporting. Status is updated in `DOCUMENTS_INGESTED` to `EXTRACTED`
on success or `EXTRACT_ERROR` on failure. The notebook is idempotent — the
`LEFT JOIN ... WHERE e.CHILD_DOC_ID IS NULL AND e.EXTRACTION_MODEL = '...'`
filter ensures already-extracted segments are skipped on re-run, and
multiple extraction models can be run against the same segments by changing
the `EXTRACTION_MODEL` constant.

## Outputs
| Table | What is written |
|---|---|
| `PROCESSING.DOCUMENTS_EXTRACTED` | One row per segment — raw extraction JSON, model, timestamp |
| `PROCESSING.DOCUMENTS_EXTRACTED_FLAT` | One row per field — value, confidence, mandatory flag, missing flag |
| `AUDIT.LLM_USAGE` | Token estimates per segment for cost reporting |
| `INGEST.DOCUMENTS_INGESTED` | STATUS updated to `EXTRACTED` or `EXTRACT_ERROR` |

## Key design decisions
- **`AI_COMPLETE` over `AI_EXTRACT`** — gives full control over prompt
  structure, failure mode handling, and null instructions per field.
  `AI_EXTRACT` was found to return placeholder text (`'See Appendix'`,
  `'None'`) as values and empty lists instead of null, which the more
  explicit `AI_COMPLETE` prompt handles correctly
- **Parallel SQL** — `AI_COMPLETE` called inside a `SELECT` with `LISTAGG`
  page concatenation; Snowflake parallelizes across rows within each doc
  type. One SQL call per doc type, not per document
- **`EXTRACTION_MODEL` on both tables** — allows A/B comparison between
  `AI_EXTRACT` and `AI_COMPLETE` output on the same documents without
  losing either result. Primary key includes `EXTRACTION_MODEL`

In [ ]:
import json
import re
import pandas as pd
from snowflake.snowpark.context import get_active_session

DB                = 'PERMAFROST_POC'
INGEST_SCHEMA     = 'INGEST'
PROCESSING_SCHEMA = 'PROCESSING'
CONFIG_SCHEMA     = 'CONFIG'
AUDIT_SCHEMA      = 'AUDIT'
EXTRACT_MODEL     = 'claude-sonnet-4-6'

def info(msg):    print(f"INFO:    {msg}")
def warning(msg): print(f"WARNING: {msg}")
def error(msg):   print(f"ERROR:   {msg}")

def estimate_tokens(text):
    return len(text) // 4 if text else 0

def is_empty(value):
    """
    Returns True if value is absent, None, empty string,
    the literal string 'None', an empty list, or an empty dict.
    """
    if value is None:
        return True
    if isinstance(value, str) and value.strip().lower() in ('none', ''):
        return True
    if isinstance(value, (list, dict)) and len(value) == 0:
        return True
    return False

def parse_ai_response(raw):
    if raw is None:
        raise ValueError("NULL response from AI_COMPLETE")
    stripped = raw.strip()
    if stripped.startswith('"') and stripped.endswith('"'):
        stripped = json.loads(stripped)
    if '```' in stripped:
        parts   = stripped.split('```')
        content = parts[1]
        if content.startswith('json'):
            content = content[4:]
        stripped = content.strip()
    result = json.loads(stripped)
    if isinstance(result, str):
        result = json.loads(result)
    if not isinstance(result, dict):
        raise ValueError(f"Expected dict, got {type(result).__name__}")
    return result

s = get_active_session()

In [ ]:
EXTRACT_PROMPT_TEMPLATE = """You are extracting structured data from a {doc_type_label} trade document for a seafood importer.

Read the ENTIRE document carefully before extracting any field.

GLOBAL RULES — apply to every field without exception:
- Return null for any field not explicitly present in the document
- NEVER return placeholder text such as 'See Appendix', 'See Attachment', 'See Annex', or any variation of 'See [location]' as a value
- If a field references an appendix and no appendix content with actual values appears in the document, return null
- NEVER infer, calculate, estimate, or guess a value
- NEVER return weight measurements (KGM, KG, LBS etc.) as lot numbers or codes
- NEVER return calendar dates as lot numbers or production codes
- NEVER confuse the consignee (always Slade Gorton, the buyer) with the supplier (the exporter)

Return ONLY valid JSON with no explanation and no markdown fences.
Each field must follow this structure:
{{
  "field_id": {{
    "value": <extracted value, list, or null>,
    "confidence": <float 0.00 to 1.00>
  }}
}}

Fields to extract:

{fields_block}

Document text:
{{page_text}}"""


def build_prompt(doc_type, mandatory_fields, optional_fields):
    """
    Builds the extraction prompt for a doc type.
    Returns a string with {{page_text}} as a placeholder
    that REPLACE() fills in SQL at extraction time.
    """
    lines = []

    lines.append("MANDATORY FIELDS (flag as missing if not found — never guess):")
    for field in mandatory_fields:
        lines.append(f"\n  Field: {field['field_id']}")
        lines.append(f"  {field['question']}")

    if optional_fields:
        lines.append("\n\nOPTIONAL FIELDS (return null if not found):")
        for field in optional_fields:
            lines.append(f"\n  Field: {field['field_id']}")
            lines.append(f"  {field['question']}")

    fields_block = '\n'.join(lines)

    return EXTRACT_PROMPT_TEMPLATE.format(
        doc_type_label = doc_type.replace('_', ' ').title(),
        fields_block   = fields_block,
    )

In [ ]:
doc_types_in_pipeline = s.sql(f"""
    SELECT DISTINCT c.DOC_TYPE
    FROM {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_CLASSIFIED c
    LEFT JOIN {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_EXTRACTED e
        ON c.CHILD_DOC_ID = e.CHILD_DOC_ID
    WHERE e.CHILD_DOC_ID IS NULL
      AND c.DOC_TYPE IS NOT NULL
      AND c.DOC_TYPE != 'unknown'
""").collect()

doc_types_needed = [row['DOC_TYPE'] for row in doc_types_in_pipeline]
config_cache     = {}

info(f"Doc types pending extraction: {doc_types_needed}")

for doc_type in doc_types_needed:
    config = s.sql(f"""
        SELECT MANDATORY_FIELDS, OPTIONAL_FIELDS
        FROM {DB}.{CONFIG_SCHEMA}.DOC_TYPE_CONFIG
        WHERE DOC_TYPE = '{doc_type}'
          AND IS_ACTIVE = TRUE
    """).collect()

    if not config:
        warning(f"  No active config for '{doc_type}' — skipping")
        continue

    mandatory = json.loads(config[0]['MANDATORY_FIELDS'] or '[]')
    optional  = json.loads(config[0]['OPTIONAL_FIELDS']  or '[]')

    if not mandatory:
        warning(f"  No mandatory fields for '{doc_type}' — skipping")
        continue

    mandatory_ids  = {f['field_id'] for f in mandatory}
    all_field_ids  = {f['field_id'] for f in mandatory + optional}
    prompt         = build_prompt(doc_type, mandatory, optional)

    config_cache[doc_type] = {
        'mandatory_ids': mandatory_ids,
        'all_field_ids': all_field_ids,
        'prompt':        prompt,
    }

    info(f"  [{doc_type}] prompt built — "
         f"{len(mandatory)} mandatory + {len(optional)} optional fields")

if not config_cache:
    print("\nNo valid configs found — nothing to extract.")

In [ ]:
PREVIEW_SIZE = 3

if config_cache:
    configured_types = ','.join(f"'{t}'" for t in config_cache.keys())
    placeholders     = ','.join(['?'] * len(config_cache))

    preview_rows = s.sql(f"""
        SELECT
            c.CHILD_DOC_ID,
            c.DOC_ID,
            c.DOC_TYPE,
            c.PAGE_START,
            c.PAGE_END,
            LISTAGG(
                '[PAGE ' || p.PAGE_NUMBER || ']' || CHR(10) ||
                COALESCE(p.PAGE_CONTENT_TRANSLATED, p.PAGE_CONTENT),
                '\\n\\n'
            ) WITHIN GROUP (ORDER BY p.PAGE_NUMBER) AS FULL_TEXT
        FROM {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_CLASSIFIED c
        JOIN {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_PAGES p
            ON  p.DOC_ID      = c.DOC_ID
            AND p.PAGE_NUMBER BETWEEN c.PAGE_START AND c.PAGE_END
        LEFT JOIN {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_EXTRACTED e
            ON c.CHILD_DOC_ID = e.CHILD_DOC_ID
            and e.EXTRACTION_MODEL = '{EXTRACT_MODEL}'
        WHERE e.CHILD_DOC_ID IS NULL
        and child_doc_id = 'db06d0f0-bdd0-43e0-aa2a-e50194bda2a1'
          AND c.DOC_TYPE IN ({configured_types})
          AND p.PAGE_CONTENT IS NOT NULL
        GROUP BY
            c.CHILD_DOC_ID, c.DOC_ID,
            c.DOC_TYPE, c.PAGE_START, c.PAGE_END
        ORDER BY c.CHILD_DOC_ID
        LIMIT {PREVIEW_SIZE}
    """).collect()

    info(f"Previewing {len(preview_rows)} document(s) — nothing written\n")

    for row in preview_rows:
        child_doc_id  = row['CHILD_DOC_ID']
        doc_type      = row['DOC_TYPE']
        full_text     = row['FULL_TEXT']
        cfg           = config_cache[doc_type]
        mandatory_ids = cfg['mandatory_ids']

        print(f"\n{'='*65}")
        print(f"  CHILD_DOC_ID : {child_doc_id}")
        print(f"  DOC_TYPE     : {doc_type}")
        print(f"  PAGES        : {row['PAGE_START']} → {row['PAGE_END']}")
        print(f"  TEXT LENGTH  : {len(full_text):,} chars")
        print(f"{'='*65}")

        try:
            # Fill page_text into the prompt
            prompt = cfg['prompt'].replace('{page_text}', full_text)

            raw = s.sql(
                f"SELECT AI_COMPLETE('{EXTRACT_MODEL}', ?) AS response",
                params=[prompt]
            ).collect()[0]['RESPONSE']

            result = parse_ai_response(raw)

            print(f"\n  {'FIELD':<30} {'VALUE':<35} {'CONF':<7} MANDATORY")
            print(f"  {'─'*30} {'─'*35} {'─'*7} {'─'*9}")

            for field_id in cfg['all_field_ids']:
                field_data = result.get(field_id, {})
                value      = field_data.get('value') \
                             if isinstance(field_data, dict) else field_data
                confidence = field_data.get('confidence') \
                             if isinstance(field_data, dict) else None
                is_mand    = field_id in mandatory_ids
                conf_str   = f"{confidence:.3f}" \
                             if confidence is not None else 'N/A'

                if is_empty(value):
                    val_str  = 'NULL'
                    mand_str = '⚠ MISSING' if is_mand else '○ empty'
                else:
                    val_str  = str(value)[:33] + '..' \
                               if len(str(value)) > 35 else str(value)
                    mand_str = '✓' if is_mand else ''

                print(f"  {field_id:<30} {val_str:<35} {conf_str:<7} {mand_str}")

            missing = [f for f in mandatory_ids
                       if is_empty(result.get(f, {}).get('value')
                                   if isinstance(result.get(f), dict)
                                   else result.get(f))]
            if missing:
                print(f"\n  ⚠  Missing mandatory: {missing}")
            else:
                print(f"\n  ✓  All mandatory fields extracted")

        except Exception as e:
            print(f"\n  ERROR: {e}")

    print(f"\n{'='*65}")
    print("Preview complete — run Step 5 when output looks correct")
    print(f"{'='*65}")


In [ ]:
#full extraction
if config_cache:
    all_extracted_rows = []
    all_flat_rows      = []
    all_llm_rows       = []
    all_errors         = []

    DOCUMENTS_CLASSIFIED = f"{DB}.{PROCESSING_SCHEMA}.DOCUMENTS_CLASSIFIED"
    DOCUMENTS_PAGES      = f"{DB}.{PROCESSING_SCHEMA}.DOCUMENTS_PAGES"
    DOCUMENTS_EXTRACTED  = f"{DB}.{PROCESSING_SCHEMA}.DOCUMENTS_EXTRACTED"
    DOCUMENTS_INGESTED   = f"{DB}.{INGEST_SCHEMA}.DOCUMENTS_INGESTED"

    def update_parent_status(child_doc_ids, status):
        if not child_doc_ids:
            return
        escaped = [str(i).replace("'", "''") for i in child_doc_ids]
        id_list = ', '.join(f"'{i}'" for i in escaped)
        s.sql(f"""
            UPDATE {DOCUMENTS_INGESTED}
            SET STATUS = '{status}'
            WHERE DOC_ID IN (
                SELECT PARENT_DOC_ID
                FROM {DOCUMENTS_CLASSIFIED}
                WHERE CHILD_DOC_ID IN ({id_list})
            )
        """).collect()

    for doc_type, cfg in config_cache.items():
        mandatory_ids = cfg['mandatory_ids']
        all_field_ids = cfg['all_field_ids']
        prompt        = cfg['prompt']

        info(f"\nExtracting '{doc_type}' - ...")

        try:
            # AI_COMPLETE runs across all CHILD_DOC_ID rows in parallel.
            results = s.sql(f"""
                SELECT
                    c.CHILD_DOC_ID,
                    c.DOC_TYPE,
                    SUM(LENGTH(COALESCE(
                        p.PAGE_CONTENT_TRANSLATED, p.PAGE_CONTENT
                    ))) AS TOTAL_CHARS,
                    AI_COMPLETE(
                        '{EXTRACT_MODEL}',
                        REPLACE(
                            ?,
                            '{{page_text}}',
                            LISTAGG(
                                '[PAGE ' || p.PAGE_NUMBER || ']' || CHR(10) ||
                                COALESCE(
                                    p.PAGE_CONTENT_TRANSLATED,
                                    p.PAGE_CONTENT
                                ),
                                '\\n\\n'
                            ) WITHIN GROUP (ORDER BY p.PAGE_NUMBER)
                        )
                    ) AS EXTRACTED
                FROM {DOCUMENTS_CLASSIFIED} c
                JOIN {DOCUMENTS_PAGES} p
                    ON  p.DOC_ID      = c.DOC_ID
                    AND p.PAGE_NUMBER BETWEEN c.PAGE_START AND c.PAGE_END
                LEFT JOIN {DOCUMENTS_EXTRACTED} e
                    ON c.CHILD_DOC_ID = e.CHILD_DOC_ID
                    AND e.EXTRACTION_MODEL = '{EXTRACT_MODEL}'
                WHERE c.DOC_TYPE     = '{doc_type}'
                  AND e.CHILD_DOC_ID  IS NULL
                  AND p.PAGE_CONTENT  IS NOT NULL
                GROUP BY c.CHILD_DOC_ID, c.DOC_TYPE
                ORDER BY c.CHILD_DOC_ID
            """, params=[prompt]).collect()

            info(f"  {len(results)} document(s) returned from Cortex")

        except Exception as exc:
            error(f"  SQL extraction failed for '{doc_type}': {exc}")
            continue

        for row in results:
            child_doc_id = row['CHILD_DOC_ID']
            raw          = row['EXTRACTED']
            total_chars  = row['TOTAL_CHARS'] or 0

            try:
                result = parse_ai_response(raw)

                #Token tracking
                input_tokens  = total_chars // 4
                output_tokens = estimate_tokens(raw)
                all_llm_rows.append({
                    'DOC_ID':        None,
                    'CHILD_DOC_ID':  child_doc_id,
                    'PIPELINE_STEP': 'EXTRACTION',
                    'MODEL':         EXTRACT_MODEL,
                    'TOKENS_IN':     input_tokens,
                    'TOKENS_OUT':    output_tokens,
                })

                #Full extraction row
                all_extracted_rows.append({
                    'CHILD_DOC_ID':     child_doc_id,
                    'DOC_TYPE':         doc_type,
                    'EXTRACTED_JSON':   json.dumps(result),
                    'EXTRACTION_MODEL': EXTRACT_MODEL,
                })

                #Flat field rows 
                for field_id in all_field_ids:
                    field_data = result.get(field_id, {})
                    value      = field_data.get('value') \
                                 if isinstance(field_data, dict) else field_data
                    confidence = field_data.get('confidence') \
                                 if isinstance(field_data, dict) else None

                    all_flat_rows.append({
                        'CHILD_DOC_ID':     child_doc_id,
                        'FIELD_ID':         field_id,
                        'FIELD_VALUE':      None if is_empty(value) else str(value),
                        'FIELD_CONFIDENCE': confidence,
                        'IS_MANDATORY':     field_id in mandatory_ids,
                        'IS_MISSING':       is_empty(value),
                        'EXTRACTION_MODEL': EXTRACT_MODEL,   
                    })

                # Missing mandatory check 
                missing = [
                    f for f in mandatory_ids
                    if is_empty(
                        result.get(f, {}).get('value')
                        if isinstance(result.get(f), dict)
                        else result.get(f)
                    )
                ]

                if missing:
                    info(f"  [OK] {child_doc_id} - "
                         f"{len(all_field_ids)} fields; "
                         f" missing mandatory: {missing}")
                else:
                    info(f"  [OK] {child_doc_id} - "
                         f"{len(all_field_ids)} fields")

            except Exception as exc:
                all_errors.append({
                    'child_doc_id': child_doc_id,
                    'doc_type':     doc_type,
                    'error':        str(exc),
                })
                error(f"  [FAIL] {child_doc_id}: {exc}")

In [ ]:
# Write DOCUMENTS_EXTRACTED

if all_extracted_rows:
    s.write_pandas(
        pd.DataFrame(all_extracted_rows),
        table_name="DOCUMENTS_EXTRACTED",
        database=DB, schema=PROCESSING_SCHEMA,
        overwrite=False,
    )
    info(f"\nWrote {len(all_extracted_rows)} row(s) to DOCUMENTS_EXTRACTED")

#  Write DOCUMENTS_EXTRACTED_FLAT

if all_flat_rows:
    s.write_pandas(
        pd.DataFrame(all_flat_rows),
        table_name="DOCUMENTS_EXTRACTED_FLAT",
        database=DB, schema=PROCESSING_SCHEMA,
        overwrite=False,
    )
    info(f"Wrote {len(all_flat_rows)} row(s) to DOCUMENTS_EXTRACTED_FLAT")

In [ ]:
# Write LLM_USAGE 

if all_llm_rows:
    s.write_pandas(
        pd.DataFrame(all_llm_rows),
        table_name="LLM_USAGE",
        database=DB, schema=AUDIT_SCHEMA,
        overwrite=False,
    )
    info(f"Wrote {len(all_llm_rows)} row(s) to LLM_USAGE")

# Update STATUS in DOCUMENTS_INGESTED

extracted_ids = [row["CHILD_DOC_ID"] for row in all_extracted_rows]
error_ids     = [row["child_doc_id"] for row in all_errors]

if extracted_ids:
    update_parent_status(extracted_ids, "EXTRACTED")
    info(f"Updated {len(extracted_ids)} document(s) to EXTRACTED")

if error_ids:
    update_parent_status(error_ids, "EXTRACT_ERROR")
    info(f"Updated {len(error_ids)} document(s) to EXTRACT_ERROR")

In [ ]:
# Summary

print(f"\n Extraction summary")
print(f"  Extracted successfully : {len(all_extracted_rows)}")
print(f"  Fields extracted       : {len(all_flat_rows)}")
print(f"  Errors                 : {len(all_errors)}")
print(f"  Total input tokens     : {sum(r['TOKENS_IN'] for r in all_llm_rows):,}")
print(f"  Total output tokens    : {sum(r['TOKENS_OUT'] for r in all_llm_rows):,}")

if all_errors:
    print("\n  Failed documents:")
    for e in all_errors:
        print(f"    {e['child_doc_id']} ({e['doc_type']}): {e['error']}")

print(f"\n Results by doc type")
s.sql(f"""
        SELECT
            c.DOC_TYPE,
            COUNT(DISTINCT e.CHILD_DOC_ID)              AS DOCS_EXTRACTED,
            COUNT(*)                                     AS TOTAL_FIELDS,
            COUNT(CASE WHEN e.IS_MISSING = TRUE
                       AND e.IS_MANDATORY THEN 1 END)   AS MISSING_MANDATORY,
            ROUND(AVG(e.FIELD_CONFIDENCE), 3)            AS AVG_CONFIDENCE
        FROM {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_EXTRACTED_FLAT e
        JOIN {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_CLASSIFIED c
            ON e.CHILD_DOC_ID = c.CHILD_DOC_ID
        GROUP BY c.DOC_TYPE
        ORDER BY c.DOC_TYPE
""").show()